# SE4050 Deep Learning 2026 - VGG16 Road-Damage Classification

**Task:** multi-label image classification of road-damage types
**(classes: `D00`, `D10`, `D20`, `D40`)** using a pretrained VGG16 backbone.

**Dataset:** road-surface image patches from the RDD-style road damage dataset
(Czech / India / Japan sources, VOC-format annotations, `label_map.pbtxt`
defines D00/D10/D20/D40). Shipped in this repository under `dataset/`.

**Why this task:** the dataset ships bounding-box annotations for damage
categories; an image-level multi-label formulation keeps every annotated image,
matches the four-class label map, and supports evidence-based model comparison
against the other architectures of the group (Custom CNN, ResNet50, ...).

**Design decisions (recorded for the report):**

| Decision | Choice | Reason |
|---|---|---|
| Task | multi-label (sigmoid + BCE) | road patches frequently contain several damage types (1300/4295 images) |
| Labeled data | train/Czech + train/India xmls (4295 images) | train/Japan labels lost in truncated upstream archive; test1/test2 ship without damage annotations |
| Split | 70 / 15 / 15, cluster-grouped | near-duplicate patches can never cross splits (leakage prevention) |
| Backbone | ImageNet-pretrained VGG16, frozen | small dataset, no GPU available (TF has no native-Windows GPU >= 2.11) |
| Training | precomputed bottleneck features + trainable head | reproducible on CPU in minutes |
| Test set | touched **once**, at the end | assignment requirement |

Every hyper-parameter comes from `.env` via `src/config.py`.

## 0. Setup, configuration and reproducibility

In [ ]:
import json
import os
import sys
import time
from pathlib import Path

# Make the project root the working directory regardless of where the kernel
# was started (repo root or notebooks/).
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score, roc_curve)

from src.config import Config
from src.utils import Timer, apply_style, load_json, save_json, set_global_seed

set_global_seed(Config.seed)
apply_style()
Config.ensure_dirs()

RUN = {}          # measured quantities collected during the run

print("TensorFlow :", tf.__version__)
print("NumPy      :", np.__version__)
print("seed       :", Config.seed)
print("classes    :", Config.class_names)
print("image size :", Config.img_size)
print("split      : test", Config.test_size, "val", Config.val_size)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
print("\nConfiguration")
print(json.dumps(Config.as_dict(), indent=2))

## 1. Dataset description, validation and EDA

The validator walks **every** split, opens **every** image (corruption check),
hashes all files (exact duplicates), parses the VOC annotations and produces
the class statistics used in the report.

In [ ]:
from src.validate_dataset import main as validate_dataset

stats = validate_dataset()
RUN["dataset"] = {
    "total_images": stats["total_images"],
    "per_split": stats["per_split"],
    "corrupt": len(stats["corrupt_images"]),
    "exact_duplicate_groups": len(stats["exact_duplicate_groups"]),
}

In [ ]:
print("images per split:", stats["per_split"])
print("image sizes     :", stats["image_sizes"])
print("corrupt images  :", stats["corrupt_images"])
print("duplicate groups:", len(stats["exact_duplicate_groups"]))
print()
print("Damage-label table (train/Czech + train/India):")
for k, v in stats["labels"].items():
    print(f"  {k}: {v}")

In [ ]:
from IPython.display import Image as IPyImage, display

for fig in ("class_distribution_splits.png",
            "damage_class_distribution.png",
            "labels_per_image.png",
            "class_distribution_by_source.png"):
    display(IPyImage(filename=str(Config.results_dir / "figures" / fig)))

## 2. Label construction and leakage-safe splitting

* keep images with **at least one** of the four label-map classes;
* compute a 64-bit perceptual dHash for every image and union near-identical
  images (Hamming <= 4) into clusters;
* assign whole clusters to **train / val / test** with per-class targets
  70 / 15 / 15, so no near-duplicate can leak across splits;
* the saved `split_manifest.csv` is shared by **all** group models.

In [ ]:
from src.dataset import create_or_load_split, leakage_report, load_xy

with Timer():
    labels_df = create_or_load_split(force=True)

leak = leakage_report(labels_df)
print("leakage report:", leak)
RUN["leakage"] = leak
assert leak["passed"], "split leakage detected"

print("\nimages per split:")
print(labels_df["split"].value_counts())
print("\nlabel counts per split (multi-label):")
print(labels_df.groupby("split")[Config.class_names].sum())
print("\nsource per split:")
print(pd.crosstab(labels_df["split"], labels_df["source"]))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
split_cls = labels_df.groupby("split")[Config.class_names].sum().loc[["train", "val", "test"]]
split_cls.plot(kind="bar", ax=ax, rot=0, colormap="tab10")
ax.set_title("Label distribution across train / val / test")
ax.set_ylabel("images containing class")
plt.tight_layout()
plt.savefig(Config.results_dir / "figures" / "split_label_distribution.png")
plt.close()

## 3. Preprocessing

| Step | Applied to | Justification |
|---|---|---|
| Resize to 224 x 224 (bilinear) | all splits | VGG16 native input size; sources ship 600/720/1024 px patches |
| `vgg16.preprocess_input` (RGB -> BGR, channel mean subtraction) | all splits | the exact normalization the ImageNet weights were trained with |
| No random augmentation | train | the backbone is frozen and features are precomputed once, so random transforms would not reach the model; documented as a limitation |
| No test-time statistics | test | test stays unseen; all decisions (thresholds) use **validation only** |
| No class re-weighting | train | imbalance is handled by per-class threshold tuning on validation (section 7) |

**Data quality issues found:** 1 truncated image (`train/Japan`), 0 byte-level
duplicates, 82 near-duplicate clusters (kept intact within one split),
`train.tar.gz` truncated upstream (labels for train/Japan and both test sets
lost) - reported, never silently patched.